# Hindsight accounting and confidence intervals

Container bootstrap (5,000 resamples) of the hindsight accounting reported in Section 4.4, and supporting comparisons with fixed timeouts.

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()

from cohort_contract import current_contract
LATEST_CONTRACT=current_contract()


## 1. Inputs and checks

In [ ]:
# Inputs and outputs stay inside the repository.
from pathlib import Path
from IPython.display import display as _cleanup_display, HTML as _CleanupHTML

RESULTS = RESULTS_REVISION
RESULTS.mkdir(parents=True, exist_ok=True)

def display(*objects, **kwargs):
    """Display-only terminology; dataframe values and calculation keys are unchanged."""
    for obj in objects:
        html = obj.data if isinstance(obj, _CleanupHTML) else (
            obj._repr_html_() if hasattr(obj, "_repr_html_") else None
        )
        if isinstance(html, str):
            html = html.replace("GPU-Enabled", "Integer-GPU")
            _cleanup_display(_CleanupHTML(html), **kwargs)
        else:
            _cleanup_display(obj, **kwargs)

# ============================================================
# 1. 입력 로드 및 container-level IR2 savings 재구성
# ============================================================

from pathlib import Path
from time import perf_counter
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display as _cleanup_display

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

def print_result_header(result_name, description, criterion):
    print("\n" + "=" * 78)
    print(f"[RESULT] {result_name}")
    print(f"설명: {description}")
    print(f"판정 기준: {criterion}")
    print("=" * 78)

def display_bilingual_table(
    frame: pd.DataFrame,
    labels: dict,
    index_labels=None,
    decimals: int = 2,
):
    shown = frame.rename(
        columns={
            key: f"{en}<br>({ko})"
            for key, (en, ko) in labels.items()
        }
    )

    if index_labels is not None:
        axes = [index_labels] if isinstance(index_labels, tuple) else index_labels
        shown = shown.rename_axis(
            index=[
                f"{en}<br>({ko})"
                for en, ko in axes
            ] if len(axes) > 1 else f"{axes[0][0]}<br>({axes[0][1]})"
        )

    html = shown.to_html(
        escape=True,
        float_format=lambda v: f"{v:,.{decimals}f}",
    ).replace("&lt;br&gt;", "<br>")

    display(HTML(html))

def normalize_filename(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .str.replace("\\", "/", regex=False)
        .str.split("/").str[-1]
        .str.strip()
        .str.replace(r"\.(csv|parquet)$", "", regex=True, case=False)
    )


DATA_ROOT = Path(str(PROCESSED_DATA))
USAGE_LOG_PATH = DATA_ROOT / "UsageLog20230401_20251130_20260301_20260831.csv"
TIMESERIES_PATH = DATA_ROOT / "TimeSeries_20240926_20251130_20260301_20260831.parquet"

TRAIN_START = pd.Timestamp("2023-03-01")
TRAIN_END = pd.Timestamp("2025-12-01")
TEST_START = pd.Timestamp("2026-03-01")
TEST_END = pd.Timestamp("2026-09-01")

usage_columns = [
    "filename",
    "created_time",
    "gpu_allocated(group)",
    "gpu_allocated(EA)",
    "cpu_allocated(Core)",
]

timeseries_columns = [
    "filename",
    "used_time(minute)",
    "gpu_allocated(group)",
    "cpu_util(%)",
    "mem_util(%)",
    "gpu_util(%)",
]

for path in [USAGE_LOG_PATH, TIMESERIES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {path}")

started = perf_counter()

usage = pd.read_csv(
    USAGE_LOG_PATH,
    usecols=usage_columns,
    low_memory=False,
)
usage["created_time"] = pd.to_datetime(
    usage["created_time"],
    errors="coerce",
)
usage["filename_clean"] = normalize_filename(usage["filename"])

if usage["created_time"].isna().any():
    raise ValueError("UsageLog created_time에 결측이 있습니다.")
if usage["filename_clean"].duplicated().any():
    raise ValueError("정규화 후 UsageLog filename 중복이 있습니다.")

usage["population"] = "Outside"
usage.loc[
    usage["created_time"].ge(TRAIN_START)
    & usage["created_time"].lt(TRAIN_END),
    "population",
] = "Train"
usage.loc[
    usage["created_time"].ge(TEST_START)
    & usage["created_time"].lt(TEST_END),
    "population",
] = "Test"

ts = pd.read_parquet(
    TIMESERIES_PATH,
    columns=timeseries_columns,
)

for col in [
    "used_time(minute)",
    "cpu_util(%)",
    "mem_util(%)",
    "gpu_util(%)",
]:
    ts[col] = pd.to_numeric(ts[col], errors="coerce")

required_numeric = [
    "used_time(minute)",
    "cpu_util(%)",
    "mem_util(%)",
    "gpu_util(%)",
]
if ts[[x for x in required_numeric if x != "gpu_util(%)"]].isna().any().any() or ts.loc[ts["gpu_allocated(group)"].ne("0.0_GPU"), "gpu_util(%)"].isna().any():
    raise ValueError("TimeSeries 필수 숫자형 컬럼에 결측이 있습니다.")

# TimeSeries container key ↔ UsageLog metadata
ts_keys = ts[["filename"]].drop_duplicates().copy()
ts_keys["filename_clean"] = normalize_filename(ts_keys["filename"])

if ts_keys["filename_clean"].duplicated().any():
    raise ValueError("정규화 후 TimeSeries filename 충돌이 있습니다.")

population = ts_keys.merge(
    usage[
        [
            "filename_clean",
            "created_time",
            "population",
            "gpu_allocated(group)",
            "gpu_allocated(EA)",
            "cpu_allocated(Core)",
        ]
    ],
    on="filename_clean",
    how="left",
    validate="one_to_one",
)

if population["population"].isna().any():
    raise ValueError("TimeSeries와 UsageLog의 container 연결 실패")

population = population.loc[
    population["population"].isin(["Train", "Test"])
].copy()

# GPU samples of GPU-allocated containers must be finite and non-negative.
if not np.isfinite(ts.loc[ts["gpu_allocated(group)"].ne("0.0_GPU"),"gpu_util(%)"]).all() or ts.loc[ts["gpu_allocated(group)"].ne("0.0_GPU"),"gpu_util(%)"].lt(0).any():
    raise ValueError("GPU utilization data contract changed")
if not ts["gpu_allocated(group)"].isin(["0.0_GPU", "1.0_GPU", "2.0_GPU", "3.0_GPU", "4.0_GPU"]).all():
    raise ValueError("Unexpected GPU group; do not redefine the cohort")
# Frozen utilization definition
is_nongpu = ts["gpu_allocated(group)"].eq("0.0_GPU")

idle = np.where(
    is_nongpu,
    (
        ts["cpu_util(%)"].le(10)
        & ts["mem_util(%)"].le(15)
    ),
    (
        ts["cpu_util(%)"].le(10)
        & ts["mem_util(%)"].le(20)
        & ts["gpu_util(%)"].eq(0)
    ),
)

ts["is_active"] = ~idle

# Container-level observation summary
grouped = ts.groupby("filename", sort=False, observed=True)

container_obs = grouped.agg(
    trace_end_minute=("used_time(minute)", "max"),
    has_active=("is_active", "max"),
    gpu_group=("gpu_allocated(group)", "first"),
    gpu_group_count=("gpu_allocated(group)", "nunique"),
)

if container_obs["gpu_group_count"].gt(1).any():
    raise ValueError("동일 container 내부에서 GPU allocation group이 변합니다.")

active = ts.loc[
    ts["is_active"],
    ["filename", "used_time(minute)"],
]

active_summary = (
    active.groupby("filename", observed=True)["used_time(minute)"]
    .agg(
        first_active_minute="min",
        last_active_minute="max",
    )
)

container_obs = container_obs.join(
    active_summary,
    how="left",
)

# 마지막 관측 sample의 active/idle 상태
tail_idx = grouped["used_time(minute)"].idxmax()
tail = (
    ts.loc[tail_idx, ["filename", "is_active"]]
    .set_index("filename")["is_active"]
)

container_obs["tail_is_idle"] = ~tail.reindex(container_obs.index)

# 분석 container metadata 결합
calc = (
    population
    .merge(
        container_obs.reset_index(),
        on="filename",
        how="left",
        validate="one_to_one",
    )
)

for col in [
    "gpu_allocated(EA)",
    "cpu_allocated(Core)",
    "trace_end_minute",
]:
    calc[col] = pd.to_numeric(calc[col], errors="coerce")

if calc[
    ["gpu_allocated(EA)", "cpu_allocated(Core)", "trace_end_minute"]
].isna().any().any():
    raise ValueError("Container-level savings 입력에 결측이 있습니다.")

if not calc["gpu_allocated(group)"].eq(calc["gpu_group"]).all():
    raise ValueError("UsageLog와 TimeSeries GPU group이 일치하지 않습니다.")

calc["container_class"] = np.where(
    calc["has_active"],
    "normal",
    "low_util",
)

calc["gpu_bin"] = np.where(
    calc["gpu_allocated(group)"].eq("0.0_GPU"),
    "Non-GPU",
    "Integer-GPU",
)

calc["post_idle_minute"] = (
    calc["trace_end_minute"]
    - calc["last_active_minute"]
).clip(lower=0)

counts = (
    pd.crosstab(
        calc["population"],
        calc["container_class"],
    )
    .reindex(
        index=["Train", "Test"],
        columns=["low_util", "normal"],
        fill_value=0,
    )
)
counts["total"] = counts.sum(axis=1)

input_passed = (
    counts.loc["Train", ["low_util", "normal", "total"]].tolist()
    == [LATEST_CONTRACT['class_counts']['Train'][k] for k in ['low_util','normal','total']]
    and counts.loc["Test", ["low_util", "normal", "total"]].tolist()
    == [LATEST_CONTRACT['class_counts']['Test'][k] for k in ['low_util','normal','total']]
    and calc["filename"].is_unique
)

print_result_header(
    "confidence_interval_input",
    "Train/Test container 단위 utilization phase와 resource allocation을 재구성",
    "Train·Test 컨테이너 수와 container 중복 0",
)

print("표: Bootstrap 입력 container 수")
display(counts)
print(f"TimeSeries rows: {len(ts):,}")
print(f"소요시간: {perf_counter() - started:,.1f}초")
print("Result:", "PASS" if input_passed else "FAIL")

if not input_passed:
    raise ValueError("CI 입력 재구성이 기존 Temporal Hold-out 결과와 일치하지 않습니다.")

del grouped, active, active_summary, tail_idx, tail
gc.collect()

# Cohort and applied IR2 timeouts (85/119/146/170) from the frozen cohort contract.
sys.path.insert(0, str(PROJECT_ROOT / "04_Analysis/revision"))
from cohort_contract import current_contract
LATEST_CONTRACT = current_contract()


## 2. Frozen IR2 savings 재현

In [ ]:
# ============================================================
# 2. Frozen timeout 적용 및 resource-minute 재현
# ============================================================

PRE_TIMEOUT = {
    "Non-GPU": 85,
    "Integer-GPU": 119,  # GPU Pre 119 (Train P90 119.0)
}

POST_TIMEOUT = {
    "Non-GPU": 146,
    "Integer-GPU": 170,
}

calc["pre_timeout_minute"] = calc["gpu_bin"].map(PRE_TIMEOUT)
calc["post_timeout_minute"] = calc["gpu_bin"].map(POST_TIMEOUT)

low_util = calc["container_class"].eq("low_util")
normal = calc["container_class"].eq("normal")

# 고정 timeout의 hindsight 회수량 정의(Eq. 2)를 사용.
normal_pre_saved = (
    calc["first_active_minute"].fillna(0)
    - 1
    - calc["pre_timeout_minute"]
).clip(lower=0)

calc["pre_saved_minute"] = np.where(
    low_util,
    (
        calc["trace_end_minute"]
        - calc["pre_timeout_minute"]
    ).clip(lower=0),
    normal_pre_saved,
)

calc["post_saved_minute"] = np.where(
    normal & calc["tail_is_idle"].eq(True),
    (
        calc["post_idle_minute"].fillna(0)
        - calc["post_timeout_minute"]
    ).clip(lower=0),
    0.0,
)

for resource, allocation_col in {
    "gpu": "gpu_allocated(EA)",
    "cpu": "cpu_allocated(Core)",
}.items():
    calc[f"{resource}_baseline"] = (
        calc[allocation_col]
        * calc["trace_end_minute"]
    )
    calc[f"{resource}_pre_saved"] = (
        calc[allocation_col]
        * calc["pre_saved_minute"]
    )
    calc[f"{resource}_post_saved"] = (
        calc[allocation_col]
        * calc["post_saved_minute"]
    )

    calc[f"{resource}_total_saved"] = (
        calc[f"{resource}_pre_saved"]
        + calc[f"{resource}_post_saved"]
    )

# Train Baseline 재현
BASELINE_REFERENCE = LATEST_CONTRACT['resources']['Train']

train = calc.loc[calc["population"].eq("Train")]

reproduction = pd.DataFrame([
    {
        "resource": "GPU",
        "baseline": train["gpu_baseline"].sum(),
        "pre_saved": train["gpu_pre_saved"].sum(),
        "post_saved": train["gpu_post_saved"].sum(),
    },
    {
        "resource": "CPU",
        "baseline": train["cpu_baseline"].sum(),
        "pre_saved": train["cpu_pre_saved"].sum(),
        "post_saved": train["cpu_post_saved"].sum(),
    },
]).set_index("resource")

reproduction["total_saved"] = (
    reproduction["pre_saved"]
    + reproduction["post_saved"]
)
reproduction["total_pct"] = (
    reproduction["total_saved"]
    / reproduction["baseline"]
    * 100
)

reproduction_passed = True
for resource in ["GPU", "CPU"]:
    for metric in ["baseline", "pre_saved", "post_saved"]:
        expected = BASELINE_REFERENCE[resource][metric]
        actual = reproduction.loc[resource, metric]
        if not np.isclose(actual, expected, rtol=0, atol=0.1):
            reproduction_passed = False

print_result_header(
    "frozen_savings_reproduction",
    "Bootstrap에 사용할 container-level contribution이 기존 Train resource savings를 재현하는지 확인",
    "GPU/CPU baseline·Pre saved·Post saved가 기존 Baseline 합계와 ±0.1 resource-minute 이내 일치",
)

display_bilingual_table(
    reproduction,
    {
        "baseline": ("Allocated resource-min", "할당 자원·분"),
        "pre_saved": ("Pre saved", "Pre 절감 자원·분"),
        "post_saved": ("Post saved", "Post 절감 자원·분"),
        "total_saved": ("Total saved", "전체 절감 자원·분"),
        "total_pct": ("Saved, %", "전체 절감률·%"),
    },
    index_labels=("Resource", "자원"),
)

print("Result:", "PASS" if reproduction_passed else "FAIL")

if not reproduction_passed:
    raise ValueError(
        "Container-level savings가 기존 Train Baseline을 재현하지 못했습니다."
    )


## 3. 95% bootstrap confidence interval

In [ ]:
# Stable container ordering before seeded container bootstrap.
calc = calc.sort_values('filename_clean',kind='stable').reset_index(drop=True)
# ============================================================
# 3. Container bootstrap: 95% CI for savings rates
# ============================================================

N_BOOT = 5_000
BOOT_BATCH = 100
BOOT_SEED = 20260926
CI_LOW = 2.5
CI_HIGH = 97.5

rng = np.random.default_rng(BOOT_SEED)

# scope별로 한 번 resample한 index를 CPU/GPU metric에서 공유한다.
SCOPE_SPECS = [
    ("All", None, ["CPU"]),
    ("Non-GPU", "Non-GPU", ["CPU"]),
    ("Integer-GPU", "Integer-GPU", ["CPU", "GPU"]),
]

def point_rates(frame, resource):
    prefix = resource.lower()
    baseline = float(frame[f"{prefix}_baseline"].sum())
    pre = float(frame[f"{prefix}_pre_saved"].sum())
    post = float(frame[f"{prefix}_post_saved"].sum())
    total = pre + post

    if baseline <= 0:
        raise ValueError(f"{resource}: baseline resource-minute가 0입니다.")

    return {
        "Pre": pre / baseline * 100,
        "Post": post / baseline * 100,
        "Total": total / baseline * 100,
    }

def bootstrap_scope(frame, resources, n_boot, batch_size, rng):
    """
    동일 container resample을 여러 resource/phase metric에 공통 적용.
    반환값: {(resource, phase): bootstrap percentage array}
    """
    n = len(frame)
    if n == 0:
        raise ValueError("Bootstrap 대상 container가 없습니다.")

    arrays = {}
    for resource in resources:
        prefix = resource.lower()
        arrays[resource] = {
            "baseline": frame[f"{prefix}_baseline"].to_numpy(dtype=float),
            "pre": frame[f"{prefix}_pre_saved"].to_numpy(dtype=float),
            "post": frame[f"{prefix}_post_saved"].to_numpy(dtype=float),
        }

    reps = {
        (resource, phase): np.empty(n_boot, dtype=float)
        for resource in resources
        for phase in ["Pre", "Post", "Total"]
    }

    written = 0

    while written < n_boot:
        b = min(batch_size, n_boot - written)
        idx = rng.integers(
            0,
            n,
            size=(b, n),
            endpoint=False,
        )

        for resource in resources:
            base = arrays[resource]["baseline"]
            pre = arrays[resource]["pre"]
            post = arrays[resource]["post"]

            boot_base = base[idx].sum(axis=1)
            boot_pre = pre[idx].sum(axis=1)
            boot_post = post[idx].sum(axis=1)

            if np.any(boot_base <= 0):
                raise ValueError(
                    f"{resource}: bootstrap replicate에서 baseline이 0 이하입니다."
                )

            reps[(resource, "Pre")][written:written+b] = (
                boot_pre / boot_base * 100
            )
            reps[(resource, "Post")][written:written+b] = (
                boot_post / boot_base * 100
            )
            reps[(resource, "Total")][written:written+b] = (
                (boot_pre + boot_post) / boot_base * 100
            )

        written += b

    return reps

ci_rows = []
bootstrap_store = {}

for population_name in ["Train", "Test"]:
    population_frame = calc.loc[
        calc["population"].eq(population_name)
    ]

    for scope_name, gpu_filter, resources in SCOPE_SPECS:
        frame = (
            population_frame
            if gpu_filter is None
            else population_frame.loc[
                population_frame["gpu_bin"].eq(gpu_filter)
            ]
        )

        # population/scope마다 seed stream을 순차적으로 사용한다.
        reps = bootstrap_scope(
            frame,
            resources=resources,
            n_boot=N_BOOT,
            batch_size=BOOT_BATCH,
            rng=rng,
        )

        for resource in resources:
            points = point_rates(frame, resource)

            for phase in ["Pre", "Post", "Total"]:
                values = reps[(resource, phase)]

                row = {
                    "population": population_name,
                    "resource": resource,
                    "gpu_group": scope_name,
                    "phase": phase,
                    "containers": len(frame),
                    "estimate_pct": points[phase],
                    "ci_low_pct": np.percentile(values, CI_LOW),
                    "ci_high_pct": np.percentile(values, CI_HIGH),
                    "bootstrap_median_pct": np.median(values),
                    "bootstrap_sd_pp": np.std(values, ddof=1),
                }
                ci_rows.append(row)

                bootstrap_store[
                    (population_name, resource, scope_name, phase)
                ] = values

ci = pd.DataFrame(ci_rows)

# Table 9와 같은 핵심 출력 순서
row_order = [
    ("GPU", "Integer-GPU"),
    ("CPU", "All"),
    ("CPU", "Non-GPU"),
    ("CPU", "Integer-GPU"),
]
phase_order = ["Pre", "Post", "Total"]

ci["resource_group_order"] = ci.apply(
    lambda r: row_order.index((r["resource"], r["gpu_group"])),
    axis=1,
)
ci["phase_order"] = ci["phase"].map(
    {p: i for i, p in enumerate(phase_order)}
)

ci = ci.sort_values(
    ["population", "resource_group_order", "phase_order"]
).reset_index(drop=True)

RESULTS = RESULTS_REVISION
RESULTS.mkdir(exist_ok=True)

ci.to_csv(
    RESULTS / "06_bootstrap_savings_ci_full.csv",
    index=False,
)

print_result_header(
    "bootstrap_savings_confidence_interval",
    "Frozen IR2 headline savings rate의 container-level bootstrap 95% CI",
    "5,000 bootstrap replicates, fixed seed, GPU integer-GPU / CPU All headline 결과 보고",
)

headline_ci = ci.loc[
    (
        ci["resource"].eq("GPU")
        & ci["gpu_group"].eq("Integer-GPU")
    )
    | (
        ci["resource"].eq("CPU")
        & ci["gpu_group"].eq("All")
    )
].copy()

headline_ci = headline_ci.loc[
    headline_ci["phase"].isin(["Pre", "Post", "Total"])
].copy()

headline_ci.to_csv(
    RESULTS / "06_bootstrap_savings_ci_headline.csv",
    index=False,
)

display_bilingual_table(
    headline_ci.set_index(
        ["population", "resource", "phase"]
    )[[
        "containers",
        "estimate_pct",
        "ci_low_pct",
        "ci_high_pct",
    ]],
    {
        "containers": ("Containers", "대상 컨테이너 수"),
        "estimate_pct": ("Estimate, %", "관측 절감률·%"),
        "ci_low_pct": ("95% CI low", "95% CI 하한·%"),
        "ci_high_pct": ("95% CI high", "95% CI 상한·%"),
    },
    index_labels=[
        ("Population", "분석 기간"),
        ("Resource", "자원"),
        ("Phase", "Pre/Post/Total"),
    ],
)

ci_passed = (
    len(ci) == 24
    and ci["estimate_pct"].between(
        ci["ci_low_pct"],
        ci["ci_high_pct"],
        inclusive="both",
    ).all()
    and (
        ci["ci_low_pct"] <= ci["ci_high_pct"]
    ).all()
)

print(f"Bootstrap repetitions: {N_BOOT:,}")
print(f"Random seed: {BOOT_SEED}")
detail_total_ci = ci.loc[ci["phase"].eq("Total")].copy()
print("[DETAIL] Total savings 95% CI by population; CPU All = Non-GPU + Integer-GPU")
display_bilingual_table(
    detail_total_ci.set_index(["population", "resource", "gpu_group"])[
        ["containers", "estimate_pct", "ci_low_pct", "ci_high_pct"]
    ],
    {"containers": ("Containers", "대상 컨테이너 수"),
     "estimate_pct": ("Estimate, %", "관측 절감률·%"),
     "ci_low_pct": ("95% CI low", "하한·%"),
     "ci_high_pct": ("95% CI high", "상한·%")},
    index_labels=[("Population", "분석 기간"), ("Resource", "자원"),
                  ("GPU type", "GPU 유형 / CPU 범위")],
)
print("[DETAIL] Pre/Post와 bootstrap SD는 06_bootstrap_savings_ci_full.csv에 저장")
print("Result:", "PASS" if ci_passed else "CHECK REQUIRED")


## 4. Train–Test 차이의 95% bootstrap interval

In [ ]:
# ============================================================
# 4. Train–Test savings-rate difference CI
# ============================================================

difference_rows = []

for resource, gpu_group in row_order:
    for phase in phase_order:
        train_values = bootstrap_store[
            ("Train", resource, gpu_group, phase)
        ]
        test_values = bootstrap_store[
            ("Test", resource, gpu_group, phase)
        ]

        diff = test_values - train_values

        train_point = float(
            ci.loc[
                ci["population"].eq("Train")
                & ci["resource"].eq(resource)
                & ci["gpu_group"].eq(gpu_group)
                & ci["phase"].eq(phase),
                "estimate_pct",
            ].iloc[0]
        )
        test_point = float(
            ci.loc[
                ci["population"].eq("Test")
                & ci["resource"].eq(resource)
                & ci["gpu_group"].eq(gpu_group)
                & ci["phase"].eq(phase),
                "estimate_pct",
            ].iloc[0]
        )

        difference_rows.append({
            "resource": resource,
            "gpu_group": gpu_group,
            "phase": phase,
            "train_estimate_pct": train_point,
            "test_estimate_pct": test_point,
            "change_pp": test_point - train_point,
            "change_ci_low_pp": np.percentile(diff, CI_LOW),
            "change_ci_high_pp": np.percentile(diff, CI_HIGH),
            "change_bootstrap_median_pp": np.median(diff),
        })

difference_ci = pd.DataFrame(difference_rows)

difference_ci["resource_group_order"] = difference_ci.apply(
    lambda r: row_order.index((r["resource"], r["gpu_group"])),
    axis=1,
)
difference_ci["phase_order"] = difference_ci["phase"].map(
    {p: i for i, p in enumerate(phase_order)}
)

difference_ci = difference_ci.sort_values(
    ["resource_group_order", "phase_order"]
).reset_index(drop=True)

difference_ci.to_csv(
    RESULTS / "06_bootstrap_train_test_difference_ci_full.csv",
    index=False,
)

print_result_header(
    "train_test_difference_confidence_interval",
    "Test−Train headline savings rate 차이의 bootstrap 95% interval",
    "GPU integer-GPU / CPU All의 Pre·Post·Total 차이를 보고",
)

headline_difference = difference_ci.loc[
    (
        difference_ci["resource"].eq("GPU")
        & difference_ci["gpu_group"].eq("Integer-GPU")
    )
    | (
        difference_ci["resource"].eq("CPU")
        & difference_ci["gpu_group"].eq("All")
    )
].copy()

headline_difference.to_csv(
    RESULTS / "06_bootstrap_train_test_difference_ci_headline.csv",
    index=False,
)

display_bilingual_table(
    headline_difference.set_index(
        ["resource", "phase"]
    )[[
        "train_estimate_pct",
        "test_estimate_pct",
        "change_pp",
        "change_ci_low_pp",
        "change_ci_high_pp",
    ]],
    {
        "train_estimate_pct": ("Train, %", "Train 절감률·%"),
        "test_estimate_pct": ("Test, %", "Test 절감률·%"),
        "change_pp": ("Change, pp", "Test−Train 차이·%p"),
        "change_ci_low_pp": ("95% CI low, pp", "차이 95% CI 하한·%p"),
        "change_ci_high_pp": ("95% CI high, pp", "차이 95% CI 상한·%p"),
    },
    index_labels=[
        ("Resource", "자원"),
        ("Phase", "Pre/Post/Total"),
    ],
)

print("[DETAIL] CPU subgroup Total differences from saved full CI")
display_bilingual_table(
    difference_ci.loc[difference_ci["phase"].eq("Total")].set_index(["resource", "gpu_group"])[
        ["train_estimate_pct", "test_estimate_pct", "change_pp", "change_ci_low_pp", "change_ci_high_pp"]
    ],
    {"train_estimate_pct": ("Train, %", "Train 절감률·%"),
     "test_estimate_pct": ("Test, %", "Test 절감률·%"),
     "change_pp": ("Change, pp", "Test−Train 차이·%p"),
     "change_ci_low_pp": ("95% CI low, pp", "차이 하한·%p"),
     "change_ci_high_pp": ("95% CI high, pp", "차이 상한·%p")},
    index_labels=[("Resource", "자원"), ("GPU type", "GPU 유형 / CPU 범위")],
)
print("Result: PASS")


In [ ]:
# Display-only replot from the existing 5,000-draw full CI results.
metric_order = [("GPU", "Integer-GPU"), ("CPU", "Non-GPU"),
                ("CPU", "Integer-GPU"), ("CPU", "All")]
total_ci = ci.loc[ci["phase"].eq("Total")].copy()
total_ci = total_ci.loc[total_ci.apply(
    lambda row: (row["resource"], row["gpu_group"]) in metric_order, axis=1)]
fig, ax = plt.subplots(figsize=(10.5, 5.8))
for row_i, (resource, group) in enumerate(metric_order):
    for period, offset, color in [("Train", -0.13, "#345b81"),
                                  ("Test", 0.13, "#d9813e")]:
        row = total_ci.loc[total_ci["population"].eq(period)
                           & total_ci["resource"].eq(resource)
                           & total_ci["gpu_group"].eq(group)].iloc[0]
        estimate = float(row["estimate_pct"])
        ax.errorbar(estimate, row_i + offset,
                    xerr=[[estimate-float(row["ci_low_pct"])],
                          [float(row["ci_high_pct"])-estimate]],
                    fmt="o", color=color, capsize=4, markersize=6,
                    label=period if row_i == 0 else None)
ax.set_yticks(range(4), ["GPU — Integer-GPU", "CPU — Non-GPU",
                            "CPU — Integer-GPU", "CPU — All monitored IR2 population"])
ax.invert_yaxis()
ax.set_xlabel("Total potentially reclaimable capacity (%)")
ax.set_title("Frozen IR2 savings: Train/Test estimates and 95% bootstrap CI")
ax.grid(axis="x", alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, title="Period")
fig.tight_layout()
figure_path = RESULTS / "06_headline_savings_ci.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
from IPython.display import Image
display(Image(filename=str(figure_path)))
plt.close(fig)
print(f"[RESULT] Figure saved: {figure_path}")


## 5. 결과 / Results

## 6. IR2와 고정 timeout 비교

In [ ]:
# [SETUP] static comparison export helpers
# The static section can also run independently in a fresh kernel.
import numpy as np
import pandas as pd
if "display" not in globals():
    from IPython.display import display
from pathlib import Path
import importlib.util
import subprocess
import sys
import tempfile
import ast

STATIC_ROOT = REVISION_CODE
STATIC_RESULTS = RESULTS_REVISION
STATIC_RESULTS.mkdir(parents=True, exist_ok=True)
STATIC_ENGINE_PATH = STATIC_ROOT / "policy_comparison.py"
REBUILD_STATIC = False
RUN_STATIC_IF_MISSING = True
STATIC_REPRESENTATIVE_TIMEOUTS = [15, 30, 45, 60, 90, 120, 150, 180, 240]
def all_candidate_summary_rows(grid, reference, selected, baseline, feasible):
    """Export existing evaluations; Train selection and replay stay unchanged.

    Grain: period x resource population x Pre/Post timeout pair, plus IR2.
    Resource-rate denominator is the same allocated resource-time for every
    policy within one period/population. Activity-return rate uses the whole
    population, not the policy-dependent number of terminated containers.
    """
    policies = {
        f"Static {int(pre)}/{int(post)} min": (i, j)
        for i, pre in enumerate(static_engine.TIMEOUTS)
        for j, post in enumerate(static_engine.TIMEOUTS)
    }
    result = static_engine.summary_rows(grid, reference, policies, baseline)
    is_ir2 = result.policy.str.startswith("IR2")
    result["policy_family"] = np.where(
        is_ir2, "IR2", np.where(
            result.pre_timeout_min.eq(result.post_timeout_min),
            "Single static", "Phase static"))
    index_by_timeout = {int(t): i for i, t in enumerate(static_engine.TIMEOUTS)}
    result["within_train_return_ceiling"] = [
        np.nan if row.policy_family == "IR2" else bool(feasible[
            index_by_timeout[int(row.pre_timeout_min)],
            index_by_timeout[int(row.post_timeout_min)]])
        for row in result.itertuples()
    ]
    for key, label in (("selected_train_single", "단일 고정 제한시간 (Single static)"),
                       ("selected_train_phase", "단계별 고정 제한시간 (Phase static)")):
        i, j = selected[label]
        result[key] = (result.pre_timeout_min.eq(int(static_engine.TIMEOUTS[i]))
                       & result.post_timeout_min.eq(int(static_engine.TIMEOUTS[j])))
    for state in ("idle", "active", "unknown"):
        for resource in ("cpu", "gpu"):
            denominator = result[f"baseline_{resource}_min"].replace(0, np.nan)
            result[f"{state}_{resource}_pct"] = (
                100 * result[f"{state}_{resource}_min"] / denominator)
    result["return_pct_of_population"] = (
        100 * result.return_containers / result.containers)
    result["terminated_pct_of_population"] = (
        100 * result.terminated / result.containers)
    expected = len(static_engine.POPS) * 3 * (len(static_engine.TIMEOUTS) ** 2 + 1)
    assert len(result) == expected
    assert not result.duplicated(
        ["population", "gpu_bin", "policy"]).any()
    assert (result.return_containers <= result.terminated).all()
    assert (result.terminated <= result.containers).all()
    assert (result.containers > 0).all()
    for resource in ("cpu", "gpu"):
        states = [f"{s}_{resource}_min" for s in ("idle", "active", "unknown")]
        assert np.isfinite(result[states].to_numpy()).all()
        assert result[states].ge(0).all().all()
        assert np.allclose(result[states].sum(axis=1),
                           result[f"gross_{resource}_min"], atol=0.05, rtol=0)
        assert result.groupby(["population", "gpu_bin"])[
            f"baseline_{resource}_min"].nunique().eq(1).all()
    return result


def static_timeout_comparison_rows(candidate_summary):
    """IR2 relative changes versus every single static timeout; no Test tuning.

    Positive resource change means more observed idle resource-time under IR2.
    It is not a joint superiority test: activity exposure is reported alongside.
    Zero static resource-time yields NaN, never an invented improvement rate.
    """
    result = candidate_summary.loc[candidate_summary.policy_family.isin(
        ["IR2", "Single static"])].copy()
    reference_columns = [
        "idle_cpu_min", "idle_gpu_min", "idle_cpu_pct", "idle_gpu_pct",
        "return_containers", "return_pct_of_population"]
    ir2 = result.loc[result.policy_family.eq("IR2"),
                     ["population", "gpu_bin", *reference_columns]].rename(
                         columns={c: f"ir2_reference_{c}" for c in reference_columns})
    result = result.merge(ir2, on=["population", "gpu_bin"],
                          how="left", validate="many_to_one")
    for resource in ("cpu", "gpu"):
        static_amount = result[f"idle_{resource}_min"]
        reference_amount = result[f"ir2_reference_idle_{resource}_min"]
        result[f"ir2_idle_{resource}_change_pct_vs_static"] = (
            100 * (reference_amount - static_amount)
            / static_amount.replace(0, np.nan))
        result[f"ir2_idle_{resource}_difference_pp_vs_static"] = (
            result[f"ir2_reference_idle_{resource}_pct"]
            - result[f"idle_{resource}_pct"])
    result["ir2_return_count_difference_vs_static"] = (
        result.ir2_reference_return_containers - result.return_containers)
    result["ir2_return_difference_pp_vs_static"] = (
        result.ir2_reference_return_pct_of_population
        - result.return_pct_of_population)
    assert len(result) == len(static_engine.POPS) * 3 * (len(static_engine.TIMEOUTS) + 1)
    return result


In [ ]:
# [RESULT] static_full_timeout_export
# Only the notebook captures the existing run's arrays; helpers are restored
# in finally. The source .py is never rewritten. Legacy outputs from the
# replay are temporary; only the two additional CSVs are written here.
static_candidate_path = STATIC_RESULTS / "06_policy_candidate_summary.csv"
static_comparison_path = STATIC_RESULTS / "06_static_timeout_comparison.csv"
static_need_run = REBUILD_STATIC or not (
    static_candidate_path.is_file() and static_comparison_path.is_file())
# Invalidate the full candidate export when the applied baseline changes.
if not static_need_run:
    retained = pd.read_csv(static_candidate_path)
    applied = retained.loc[retained.policy.str.startswith("IR2") & retained.gpu_bin.eq("Non-GPU")]
    static_need_run = len(applied) != 2 or not applied.pre_timeout_min.eq(LATEST_CONTRACT["frozen_pre"]["Non-GPU"]).all()
    if static_need_run:
        print("[RESULT] Full candidate export regenerated for current85-minute applied baseline")
static_missing = []
if static_need_run and RUN_STATIC_IF_MISSING:
    if not STATIC_ENGINE_PATH.is_file():
        static_missing.append(str(STATIC_ENGINE_PATH))
    if not (STATIC_ROOT / "cache_paths.py").is_file():
        static_missing.append("cache_paths.py (기존 프로젝트의 모듈)")
    if importlib.util.find_spec("pyarrow") is None:
        static_missing.append("pyarrow (기존 분석 Python 환경)")
    if STATIC_ENGINE_PATH.is_file():
        work_root = STATIC_ROOT
        for relative in ["online_ir2/simulation.py", "threshold_sensitivity/sensitivity.py"]:
            if not (work_root / relative).is_file():
                static_missing.append(str(work_root / relative))
    if static_missing:
        print("[RESULT] static_full_timeout_export: BLOCKED")
        print("전체 트레이스 계산에 필요한 입력/모듈:")
        for item in static_missing:
            print(" -", item)
        print("기존 150분 CSV 검산만 수행하며 timeout별 결과를 만들어내지 않는다.")
    else:
        sys.path.insert(0, str(STATIC_ROOT))
        spec = importlib.util.spec_from_file_location("fgcs_static_engine", STATIC_ENGINE_PATH)
        static_engine = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(static_engine)
        static_context = {}
        static_original_select = static_engine.select_policies
        static_original_summary = static_engine.summary_rows
        def static_capture_select(grid, reference):
            selected, feasible = static_original_select(grid, reference)
            static_context["feasible"] = feasible
            return selected, feasible
        def static_capture_summary(grid, reference, selected, baseline):
            static_context.update(grid=grid, reference=reference,
                                  selected=selected, baseline=baseline)
            return static_original_summary(grid, reference, selected, baseline)
        try:
            static_engine.select_policies = static_capture_select
            static_engine.summary_rows = static_capture_summary
            with tempfile.TemporaryDirectory(prefix="static_replay_", dir=STATIC_RESULTS) as replay_dir:
                static_run_validation = static_engine.run(output=Path(replay_dir))
        finally:
            static_engine.select_policies = static_original_select
            static_engine.summary_rows = static_original_summary
        assert static_run_validation["status"] == "PASS"
        static_candidates = all_candidate_summary_rows(**static_context)
        static_comparison = static_timeout_comparison_rows(static_candidates)
        static_candidates.to_csv(static_candidate_path, index=False,
                                 encoding="utf-8-sig", float_format="%.10f")
        static_comparison.to_csv(static_comparison_path, index=False,
                                 encoding="utf-8-sig", float_format="%.10f")
        print("[RESULT] static_full_timeout_export: PASS")
        print(f"Candidates: {len(static_candidates):,} rows; single static: {len(static_comparison):,} rows")
        del static_context
elif static_need_run:
    print("[RESULT] static_full_timeout_export: NOT RUN (RUN_STATIC_IF_MISSING=False)")
else:
    print("[RESULT] static_full_timeout_export: using existing full CSVs")


### 6.1 회수 자원시간·회수율·IR2 상대 증감률

In [ ]:
# [RESULT] static_150_resource_comparison
static_full_available = static_candidate_path.is_file() and static_comparison_path.is_file()
if static_full_available:
    static_candidates = pd.read_csv(static_candidate_path)
    static_comparison = pd.read_csv(static_comparison_path)
    assert len(static_candidates) == 3180 and len(static_comparison) == 144
    assert not static_candidates.duplicated(["population", "gpu_bin", "policy"]).any()
    for resource in ["cpu", "gpu"]:
        assert static_candidates.groupby(["population", "gpu_bin"])[f"baseline_{resource}_min"].nunique().eq(1).all()
    static_150 = static_comparison.loc[static_comparison.pre_timeout_min.eq(150)].copy()
else:
    legacy_path = STATIC_RESULTS / "06_policy_summary.csv"
    if not legacy_path.is_file():
        raise FileNotFoundError(f"기존 150분 결과도 없습니다: {legacy_path}")
    legacy = pd.read_csv(legacy_path)
    static_150 = legacy.loc[legacy.policy.str.contains("Single static", regex=False)].copy()
    reference = legacy.loc[legacy.policy.str.startswith("IR2")].set_index(["population", "gpu_bin"])
    assert len(static_150) == 6 and reference.index.is_unique
    for resource in ["cpu", "gpu"]:
        ir2_values = [reference.loc[(r.population, r.gpu_bin), f"idle_{resource}_min"]
                      for r in static_150.itertuples()]
        static_150[f"ir2_reference_idle_{resource}_min"] = ir2_values
        denom = static_150[f"baseline_{resource}_min"].replace(0, np.nan)
        static_150[f"idle_{resource}_pct"] = 100 * static_150[f"idle_{resource}_min"] / denom
        static_150[f"ir2_reference_idle_{resource}_pct"] = 100 * static_150[f"ir2_reference_idle_{resource}_min"] / denom
        static_150[f"ir2_idle_{resource}_change_pct_vs_static"] = (
            100 * (static_150[f"ir2_reference_idle_{resource}_min"] - static_150[f"idle_{resource}_min"])
            / static_150[f"idle_{resource}_min"].replace(0, np.nan))
    print("전체 timeout CSV 미생성: 아래는 기존 150분 결과만 검산한 표다.")

static_metrics = [("GPU-Enabled", "gpu"), ("Non-GPU", "cpu"),
                  ("GPU-Enabled", "cpu"), ("All", "cpu")]
static_resource_rows = []
for period in ["Train", "Test"]:
    for group, resource in static_metrics:
        row = static_150.loc[static_150.population.eq(period) & static_150.gpu_bin.eq(group)]
        assert len(row) == 1
        r = row.iloc[0]
        static_resource_rows.append({
            "Period": period, "Population": "Integer-GPU" if group == "GPU-Enabled" else group,
            "Resource": "GPU, GPU·min" if resource == "gpu" else "CPU, core·min",
            "IR2 idle resource-time": r[f"ir2_reference_idle_{resource}_min"],
            "Static idle resource-time": r[f"idle_{resource}_min"],
            "IR2 idle rate, %": r[f"ir2_reference_idle_{resource}_pct"],
            "Static idle rate, %": r[f"idle_{resource}_pct"],
            "IR2 relative change, %": r[f"ir2_idle_{resource}_change_pct_vs_static"],
        })
static_resource_table = pd.DataFrame(static_resource_rows)
static_resource_table.to_csv(STATIC_RESULTS / "06_static_150_resource_comparison.csv",
                             index=False, encoding="utf-8-sig", float_format="%.10f")
print("[RESULT] static_150_resource_comparison")
print("표: 정책별 관측 유휴 자원시간·회수율과 IR2 상대 증감률")
labels = {"Period": "Period / 기간", "Population": "Population / 구분", "Resource": "Resource / 자원·단위",
          "IR2 idle resource-time": "IR2 idle time / IR2 관측 유휴 자원시간",
          "Static idle resource-time": "Static idle time / Static 관측 유휴 자원시간",
          "IR2 idle rate, %": "IR2 rate, % / IR2 회수율",
          "Static idle rate, %": "Static rate, % / Static 회수율",
          "IR2 relative change, %": "IR2 change, % / IR2 상대 증감률"}
display(static_resource_table.rename(columns=labels).round(2))
print("Result: PASS (기존 150분 결과의 비율·분모 계산)")


### 6.2 고정 timeout별 자원 회수 비교표와 이후 활동 보조 표

In [ ]:
# [RESULT] static_timeout_resource_and_activity_tables
if not static_full_available:
    print("[RESULT] timeout별 Train/Test 비교표: NOT GENERATED")
    print("6절을 먼저 실행해 전체 timeout CSV를 생성한다.")
else:
    static_resource_display_rows = []
    for group, resource in static_metrics:
        group_name = "Integer-GPU" if group == "GPU-Enabled" else group
        rows = static_comparison.loc[
            static_comparison.gpu_bin.eq(group)
            & static_comparison.policy_family.eq("Single static")
            & static_comparison.pre_timeout_min.isin(STATIC_REPRESENTATIVE_TIMEOUTS)]
        output_rows = []
        for period in ["Train", "Test"]:
            points = rows.loc[rows.population.eq(period)].sort_values("pre_timeout_min")
            assert len(points) == len(STATIC_REPRESENTATIVE_TIMEOUTS)
            for r in points.itertuples():
                record = {
                    "Period": period,
                    "Timeout, min": int(r.pre_timeout_min),
                    "IR2 idle resource-time": getattr(r, f"ir2_reference_idle_{resource}_min"),
                    "Static idle resource-time": getattr(r, f"idle_{resource}_min"),
                    "IR2 idle rate, %": getattr(r, f"ir2_reference_idle_{resource}_pct"),
                    "Static idle rate, %": getattr(r, f"idle_{resource}_pct"),
                    "IR2 relative change, %": getattr(r, f"ir2_idle_{resource}_change_pct_vs_static"),
                }
                output_rows.append(record)
                static_resource_display_rows.append({
                    "Population": group_name, "Resource": resource.upper(),
                    "Unit": "GPU·min" if resource == "gpu" else "core·min", **record})
        unit = "GPU·min" if resource == "gpu" else "core·min"
        print(f"표: {group_name} {resource.upper()}의 고정 timeout별 관측 유휴 자원시간({unit})·회수율 비교")
        resource_labels = {
            "Period": "Period / 기간", "Timeout, min": "Timeout, min / 고정 시간",
            "IR2 idle resource-time": "IR2 idle time / IR2 관측 유휴 자원시간",
            "Static idle resource-time": "Static idle time / Static 관측 유휴 자원시간",
            "IR2 idle rate, %": "IR2 rate, % / IR2 회수율",
            "Static idle rate, %": "Static rate, % / Static 회수율",
            "IR2 relative change, %": "IR2 change, % / IR2 상대 증감률",
        }
        display(pd.DataFrame(output_rows).rename(columns=resource_labels).round(2))
    pd.DataFrame(static_resource_display_rows).to_csv(
        STATIC_RESULTS / "06_static_timeout_resource_comparison_table.csv",
        index=False, encoding="utf-8-sig", float_format="%.10f")

    # Activity is a separate supporting table. Its rate uses the whole
    # period/population so the denominator does not change with timeout.
    static_activity_rows = []
    for group in ["Non-GPU", "GPU-Enabled"]:
        group_name = "Integer-GPU" if group == "GPU-Enabled" else group
        records = []
        for period in ["Train", "Test"]:
            rows = static_comparison.loc[
                static_comparison.population.eq(period) & static_comparison.gpu_bin.eq(group)]
            ref = rows.loc[rows.policy_family.eq("IR2")].iloc[0]
            records.append({"Period": period, "Policy": "IR2", "Timeout, min": "유형·단계별",
                            "Population containers": int(ref.containers),
                            "Later-active containers": int(ref.return_containers),
                            "Later-active population rate, %": ref.return_pct_of_population})
            points = rows.loc[rows.policy_family.eq("Single static")
                              & rows.pre_timeout_min.isin(STATIC_REPRESENTATIVE_TIMEOUTS)].sort_values("pre_timeout_min")
            assert len(points) == len(STATIC_REPRESENTATIVE_TIMEOUTS)
            for r in points.itertuples():
                records.append({"Period": period, "Policy": "Static",
                                "Timeout, min": int(r.pre_timeout_min),
                                "Population containers": int(r.containers),
                                "Later-active containers": int(r.return_containers),
                                "Later-active population rate, %": r.return_pct_of_population})
        print(f"보조 표: {group_name}의 종료 판단 이후 활동 컨테이너 수와 비율")
        activity_labels = {
            "Period": "Period / 기간", "Policy": "Policy / 정책",
            "Timeout, min": "Timeout, min / 고정 시간",
            "Population containers": "Population / 전체 컨테이너 수",
            "Later-active containers": "Later active / 종료 후 활동 컨테이너 수",
            "Later-active population rate, %": "Later active, % / 전체 중 종료 후 활동 비율",
        }
        display(pd.DataFrame(records).rename(columns=activity_labels).round(2))
        static_activity_rows.extend({"Population": group_name, **r} for r in records)
    pd.DataFrame(static_activity_rows).to_csv(
        STATIC_RESULTS / "06_static_timeout_activity_comparison_table.csv",
        index=False, encoding="utf-8-sig", float_format="%.10f")
    print("Result: PASS (Train/Test 대표 timeout별 주 표와 보조 표 생성)")


### 6.3 Static timeout 변화에 따른 회수율과 IR2 상대 증감률

In [ ]:
# [RESULT] static_timeout_resource_curves
if not static_full_available:
    print("[RESULT] static timeout 선그래프: NOT GENERATED")
    print("6절을 먼저 실행해 전체 timeout CSV를 생성한다.")
else:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter

    curve_metrics = [("Non-GPU", "cpu", "Non-GPU CPU"),
                     ("GPU-Enabled", "cpu", "Integer-GPU CPU"),
                     ("GPU-Enabled", "gpu", "Integer-GPU GPU")]
    curve_styles = {"Train": {"color": "#2563A6", "marker": "o"},
                    "Test": {"color": "#C27812", "marker": "s"}}
    curve_points = {}
    for group, resource, title in curve_metrics:
        for period in ["Train", "Test"]:
            points = static_comparison.loc[
                static_comparison.population.eq(period)
                & static_comparison.gpu_bin.eq(group)
                & static_comparison.policy_family.eq("Single static")
                & static_comparison.pre_timeout_min.isin(STATIC_REPRESENTATIVE_TIMEOUTS)
            ].sort_values("pre_timeout_min").copy()
            assert points.pre_timeout_min.tolist() == sorted(STATIC_REPRESENTATIVE_TIMEOUTS)
            assert points.post_timeout_min.eq(points.pre_timeout_min).all()
            columns = [f"idle_{resource}_pct", f"ir2_reference_idle_{resource}_pct",
                       f"ir2_idle_{resource}_change_pct_vs_static"]
            assert np.isfinite(points[columns].to_numpy(dtype=float)).all()
            assert points[f"ir2_reference_idle_{resource}_pct"].nunique() == 1
            baseline = points[f"baseline_{resource}_min"]
            assert baseline.gt(0).all() and baseline.nunique() == 1
            np.testing.assert_allclose(points[f"idle_{resource}_pct"],
                100 * points[f"idle_{resource}_min"] / baseline)
            np.testing.assert_allclose(points[f"ir2_idle_{resource}_change_pct_vs_static"],
                100 * (points[f"ir2_reference_idle_{resource}_min"] - points[f"idle_{resource}_min"])
                / points[f"idle_{resource}_min"])
            curve_points[(group, resource, period)] = points

    curve_outputs = [
        ("rate", "Observed idle resource recovery rate", "Recovery rate (%)",
         "06_static_timeout_resource_rate_curves.png"),
        ("change", "IR2 relative change compared with static", "IR2 relative change (%)",
         "06_static_timeout_relative_change_curves.png"),
    ]
    for kind, heading, ylabel, filename in curve_outputs:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
        for ax, (group, resource, title) in zip(axes, curve_metrics):
            for period, style in curve_styles.items():
                points = curve_points[(group, resource, period)]
                x = points.pre_timeout_min.to_numpy(dtype=float)
                if kind == "rate":
                    y = points[f"idle_{resource}_pct"].to_numpy(dtype=float)
                    ax.plot(x, y, linewidth=2, markersize=5,
                            label=f"Static {period}", **style)
                    ax.axhline(points[f"ir2_reference_idle_{resource}_pct"].iloc[0],
                               color=style["color"], linestyle="--", linewidth=1.6,
                               label=f"IR2 {period}")
                else:
                    y = points[f"ir2_idle_{resource}_change_pct_vs_static"].to_numpy(dtype=float)
                    ax.plot(x, y, linewidth=2, markersize=5,
                            label=period, **style)
            if kind == "change":
                ax.axhline(0, color="#666666", linewidth=1, linestyle="--")
            ax.set_title(title, fontsize=12)
            ax.set_xlabel("Static timeout (min)")
            ax.set_xticks(sorted(STATIC_REPRESENTATIVE_TIMEOUTS))
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
            ax.grid(axis="y", alpha=0.25)
            ax.spines[["top", "right"]].set_visible(False)
            ax.margins(x=0.04)
        if kind == "rate":
            axes[0].set_ylim(0, 100)
        else:
            # Same y scale across all three panels, including the 0% reference.
            limits = [0.0]
            for (group, resource, period), points in curve_points.items():
                limits.extend(points[f"ir2_idle_{resource}_change_pct_vs_static"].tolist())
            lower, upper = min(limits), max(limits)
            pad = max((upper - lower) * 0.08, 1.0)
            axes[0].set_ylim(lower - pad, upper + pad)
        axes[0].set_ylabel(ylabel)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False)
        fig.suptitle(heading, fontsize=14)
        fig.subplots_adjust(left=0.075, right=0.99, bottom=0.21, top=0.80, wspace=0.12)
        save_path = STATIC_RESULTS / filename
        fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor="white")
        plt.show()
        plt.close(fig)
        print(f"Saved: {save_path}")
    print("Result: PASS (대표 timeout별 표와 3패널 선그래프, 동일 분모·비율 검산)")


### 6.4 해석과 확인 범위

## 7. Idle lifecycle: Pre / Intermediate / Post별 시간 분포

In [ ]:
# 7.1 Setup + phase accounting helpers (fresh kernel OK)
from pathlib import Path
import importlib.util
import sys
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

PHASE_ROOT = REVISION_CODE  # policy_comparison.py와 같은 폴더
PHASE_RESULTS = RESULTS_REVISION
PHASE_ORDER = ['Pre', 'Intermediate', 'Post', 'Never-active']
PHASE_REBUILD = True  # 원본 trace를 재계산. 완료 후 False로 바꾸면 검증된 CSV 재사용.
PHASE_EXPECTED_COUNTS = {(pop, 'Integer-GPU' if b=='GPU-Enabled' else b): count for pop,groups in LATEST_CONTRACT['group_counts'].items() for b,count in groups.items()}

def phase_support(times, states, active_tail, idle_tail):
    """Invert the existing engine's suffix sums; never fill missing time."""
    times = np.asarray(times, dtype=np.int64)
    states = np.asarray(states, dtype=str)
    n = len(times)
    if not n or len(states) != n or np.any(np.diff(times) <= 0):
        raise AssertionError('Empty / unordered / mismatched trace')
    weights = {}
    for name, suffix in [('active', active_tail), ('idle', idle_tail)]:
        suffix = np.asarray(suffix, dtype=float)
        if len(suffix) != n or not np.isfinite(suffix).all():
            raise AssertionError('Suffix support shape / finiteness changed')
        w = suffix - np.r_[suffix[1:], 0.0]
        if np.any(w < -1e-6) or np.any(w[states != name] > 1e-6):
            raise AssertionError('Engine support attribution changed; inspect first_crossing')
        weights[name] = np.maximum(w, 0)
    dt = np.r_[np.diff(times) / 60e9, 0.0]
    if np.any(weights['active'] + weights['idle'] > dt + 1e-6):
        raise AssertionError('Observed support exceeds elapsed interval')
    weights['unknown'] = np.maximum(dt - weights['active'] - weights['idle'], 0)
    active_indices = np.flatnonzero(states == 'active')
    labels = np.full(n, 'Never-active', dtype=object)
    if len(active_indices):
        first, last = int(active_indices[0]), int(active_indices[-1])
        labels[:first] = 'Pre'
        labels[first:last + 1] = 'Intermediate'
        labels[last + 1:] = 'Post'
    total = sum(weights['idle'][labels == phase].sum() for phase in PHASE_ORDER)
    assert np.isclose(total, np.asarray(idle_tail)[0], atol=1e-6)
    assert np.isclose(sum(w.sum() for w in weights.values()), dt.sum(), atol=1e-6)
    return labels, weights, active_indices

def phase_events(times, states, labels, weights, active_indices):
    """Pre/Post windows and returned Intermediate gaps, separate from idle support.

    Intermediate span: first observed idle after activity -> next observed activity.
    Unknown inside that span remains separately quantified. Event span percentiles
    must not be confused with per-container accumulated idle-support percentiles.
    """
    times = np.asarray(times, dtype=np.int64)
    states = np.asarray(states, dtype=str)
    windows = []
    n = len(times)
    if not len(active_indices):
        windows.append(('Never-active', 0, n, n - 1, False))
    else:
        first, last = int(active_indices[0]), int(active_indices[-1])
        windows.append(('Pre', 0, first, first, True))
        for left, right in zip(active_indices[:-1], active_indices[1:]):
            windows.append(('Intermediate', int(left) + 1, int(right), int(right), True))
        windows.append(('Post', last + 1, n, n - 1, False))
    events = []
    for phase, start, stop, end, returns in windows:
        idle_indices = np.flatnonzero(states[start:stop] == 'idle') + start
        if not len(idle_indices):
            continue
        # Start at first observed idle; only idle time is charged to this phase.
        start = int(idle_indices[0])
        idle = float(weights['idle'][start:stop].sum())
        span = float((times[end] - times[start]) / 60e9)
        active = float(weights['active'][start:stop].sum())
        unknown = max(span - idle - active, 0.0)
        assert np.isclose(idle + active + unknown, span, atol=1e-6)
        events.append(dict(phase=phase, span_min=span, idle_min=idle,
                           unknown_min=unknown, observed_return=returns))
    for phase in PHASE_ORDER:
        assert np.isclose(sum(e['idle_min'] for e in events if e['phase'] == phase),
                          weights['idle'][labels == phase].sum(), atol=1e-6)
    return events

def phase_first_hit(pre, post):
    hits = [int(x) for x in (pre, post) if x >= 0]
    return min(hits) if hits else -1

def phase_quantiles(values):
    x = np.asarray(values, dtype=float)
    if not len(x):
        return {k: np.nan for k in ['P50', 'P90', 'P95', 'P99', 'Max']}
    return dict(zip(['P50', 'P90', 'P95', 'P99', 'Max'],
                    [*np.quantile(x, [.5, .9, .95, .99]), x.max()]))

def phase_top_share(values, fraction):
    x = np.sort(np.asarray(values, dtype=float))[::-1]
    return (100 * x[:max(1, int(np.ceil(len(x) * fraction)))].sum() / x.sum()
            if x.sum() > 0 else np.nan)


In [ ]:
# 7.2 Synthetic checks only — these numbers are NOT cluster measurements.
def phase_logic_checks():
    def check(states, expected, times=None, idle_weights=None, active_weights=None):
        n = len(states)
        t = np.arange(n, dtype=np.int64) * 60_000_000_000 if times is None else np.array(times, dtype=np.int64) * 60_000_000_000
        dt = np.r_[np.diff(t) / 60e9, 0.0]
        idle = dt * (np.array(states) == 'idle') if idle_weights is None else np.array(idle_weights, float)
        active = dt * (np.array(states) == 'active') if active_weights is None else np.array(active_weights, float)
        suffix = lambda x: np.cumsum(x[::-1])[::-1]
        labels, w, ai = phase_support(t, states, suffix(active), suffix(idle))
        result = {p: float(w['idle'][labels == p].sum()) for p in PHASE_ORDER}
        assert result == dict(zip(PHASE_ORDER, expected)), result
        events = phase_events(t, states, labels, w, ai)
        return labels, w, ai, events
    check(['idle', 'active', 'idle', 'idle', 'active', 'idle', 'idle'], [1, 2, 1, 0])
    check(['idle', 'idle', 'idle'], [0, 0, 0, 2])
    check(['active', 'active', 'active'], [0, 0, 0, 0])
    _, w, _, events = check(['active', 'idle', 'idle', 'active', 'idle'],
        [0, 2, 0, 0], times=[0, 1, 100, 101, 102],
        idle_weights=[0, 1, 1, 0, 0], active_weights=[1, 0, 0, 1, 0])
    assert w['unknown'].sum() == 98
    gap = next(e for e in events if e['phase'] == 'Intermediate')
    assert gap['span_min'] == 100 and gap['idle_min'] == 2 and gap['unknown_min'] == 98
    check(['idle', 'unknown', 'active', 'unknown', 'idle', 'active', 'idle', 'idle'], [1, 1, 1, 0])
    assert phase_first_hit(-1, 4) == 4 and phase_first_hit(3, 4) == 3
    assert phase_first_hit(-1, -1) == -1
    assert phase_top_share([9, 1, 0, 0], .25) == 90
    # After a trigger, later final idle belongs to Post even if trigger was Intermediate.
    labels, w, _, _ = check(['active', 'idle', 'active', 'idle', 'idle'], [0, 1, 1, 0])
    assert w['idle'][1:][labels[1:] == 'Post'].sum() == 1
    return 'PASS: phase partition, never-active, missing-time preservation, return-gap span, post-trigger attribution'

print('[CHECK]', phase_logic_checks())


In [ ]:
# 7.3 Load the original engine and stream raw trace; preserve its support rules.
PHASE_RESULTS.mkdir(parents=True, exist_ok=True)
phase_paths = {name: PHASE_RESULTS / f'06_idle_phase_{name}.csv'
               for name in ['containers', 'events', 'policy_containers', 'coverage']}
phase_provenance_path = PHASE_RESULTS / '06_idle_phase_provenance.json'
phase_engine_path = PHASE_ROOT / 'policy_comparison.py'
if not phase_engine_path.is_file():
    raise FileNotFoundError(f'기존 Mac 프로젝트 폴더에서 실행하세요: {phase_engine_path}')
sys.path.insert(0, str(PHASE_ROOT))
phase_spec = importlib.util.spec_from_file_location('fgcs_phase_engine', phase_engine_path)
phase_engine = importlib.util.module_from_spec(phase_spec)
phase_spec.loader.exec_module(phase_engine)

def phase_fingerprint(path):
    path = Path(path)
    stat = path.stat()
    return dict(path=str(path.resolve()), size_bytes=stat.st_size, mtime_ns=stat.st_mtime_ns)

phase_usage = phase_engine.load_usage()
assert phase_usage.index.is_unique
phase_inputs = dict(method='suffix-support-phase-v1',
    trace=phase_fingerprint(phase_engine.TRACE),
    engine_sha256=hashlib.sha256(phase_engine_path.read_bytes()).hexdigest(),
    simulation=phase_fingerprint(Path(phase_engine.prepare_trace.__code__.co_filename)),
    sensitivity=phase_fingerprint(Path(phase_engine.first_crossing.__code__.co_filename)),
    usage_sha256=hashlib.sha256(phase_usage.sort_index().to_csv().encode()).hexdigest(),
    pre={k: int(v) for k, v in phase_engine.PRE.items()},
    post={k: int(v) for k, v in phase_engine.POST.items()}, static_timeout=150)
phase_cache_ok = (all(p.is_file() for p in phase_paths.values()) and phase_provenance_path.is_file())
if phase_cache_ok:
    phase_cache_ok = json.loads(phase_provenance_path.read_text()) == phase_inputs

if PHASE_REBUILD or not phase_cache_ok:
    import pyarrow.parquet as pq
    phase_file = pq.ParquetFile(phase_engine.TRACE)
    phase_columns = ['filename', 'time', 'used_time(minute)', 'gpu_allocated(group)',
                     'cpu_util(%)', 'mem_util(%)', 'gpu_util(%)']
    container_rows, event_rows, policy_rows, coverage_rows = [], [], [], []
    seen = set()
    for row_group in range(phase_file.num_row_groups):
        frame = phase_file.read_row_group(row_group, columns=phase_columns).to_pandas()
        names = frame.filename.dropna().unique()
        assert len(names) == 1, f'One container per row group required: {row_group}'
        key = phase_engine.normalize_filename(names[0])
        assert key not in seen and key in phase_usage.index, f'Duplicate / unmatched container: {key}'
        seen.add(key)
        meta = phase_usage.loc[key]
        assert meta.population in ['Train', 'Test']
        engine_bin = 'Non-GPU' if meta['gpu_allocated(group)'] == '0.0_GPU' else 'GPU-Enabled'
        group = 'Non-GPU' if engine_bin == 'Non-GPU' else 'Integer-GPU'
        cpu, gpu = float(meta['cpu_allocated(Core)']), float(meta['gpu_allocated(EA)'])
        assert np.isfinite([cpu, gpu]).all() and cpu >= 0 and gpu >= 0
        assert group == 'Non-GPU' or gpu >= 1 and np.isclose(gpu, round(gpu))
        trace, audit = phase_engine.prepare_trace(frame, engine_bin, meta.terminated_time)
        assert not audit['duplicate_rows'] and not audit['rows_after_termination']
        pre_limit, post_limit = int(phase_engine.PRE[engine_bin]), int(phase_engine.POST[engine_bin])
        times, pre_hits, post_hits, at, it, ai = phase_engine.first_crossing(
            trace, {t: t for t in set([150, pre_limit])},
            {t: t for t in set([150, post_limit])})
        # Refuse to silently change alignment or the engine's support semantics.
        assert np.array_equal(np.asarray(times), pd.to_datetime(trace.time).to_numpy(dtype='datetime64[ns]').astype('int64'))
        labels, weights, active_indices = phase_support(times, trace.state, at, it)
        assert np.array_equal(ai, active_indices)
        base = dict(filename=key, population=meta.population, gpu_bin=group)
        for phase in PHASE_ORDER:
            amount = float(weights['idle'][labels == phase].sum())
            container_rows.append({**base, 'phase': phase, 'idle_min': amount,
                                   'idle_cpu_min': amount * cpu, 'idle_gpu_min': amount * gpu})
        for event_id, event in enumerate(phase_events(times, trace.state, labels, weights, ai)):
            event_rows.append({**base, 'event_id': event_id, **event})
        coverage_rows.append({**base, 'has_active': bool(len(ai)),
            'span_min': float((times[-1] - times[0]) / 60e9),
            'baseline_min': float(trace.elapsed.max()),
            'active_min': float(weights['active'].sum()),
            'idle_min': float(weights['idle'].sum()),
            'unknown_min': float(weights['unknown'].sum())})
        for policy, pre, post in [('IR2', pre_limit, post_limit), ('Static 150', 150, 150)]:
            hit = phase_first_hit(pre_hits[pre], post_hits[post])
            next_activity = ai[ai > hit] if hit >= 0 else np.array([], dtype=int)
            episode_stop = int(next_activity[0]) if len(next_activity) else len(times)
            episode_idle = float(weights['idle'][hit:episode_stop].sum()) if hit >= 0 else 0
            active_after = float(at[hit]) if hit >= 0 else 0
            amounts = {p: (float(weights['idle'][hit:][labels[hit:] == p].sum()) if hit >= 0 else 0)
                       for p in PHASE_ORDER}
            expected_idle = float(it[hit]) if hit >= 0 else 0
            assert np.isclose(sum(amounts.values()), expected_idle, atol=1e-6)
            for phase, amount in amounts.items():
                policy_rows.append({**base, 'policy': policy, 'phase': phase,
                    'trigger_phase': str(labels[hit]) if hit >= 0 else 'No trigger',
                    'terminated': int(hit >= 0),
                    'later_active': int(hit >= 0 and len(ai) > 0 and ai[-1] > hit),
                    'trigger_elapsed_min': float(trace.elapsed.iloc[hit]) if hit >= 0 else np.nan,
                    'episode_idle_cpu_min': episode_idle * cpu,
                    'episode_idle_gpu_min': episode_idle * gpu,
                    'active_after_cpu_min': active_after * cpu,
                    'active_after_gpu_min': active_after * gpu,
                    'idle_min': amount, 'idle_cpu_min': amount * cpu, 'idle_gpu_min': amount * gpu,
                    'baseline_cpu_min': cpu * float(trace.elapsed.max()),
                    'baseline_gpu_min': gpu * float(trace.elapsed.max())})
        if (row_group + 1) % 2500 == 0:
            print(f'[PROGRESS] {row_group + 1:,}/{phase_file.num_row_groups:,}', flush=True)
    phase_containers = pd.DataFrame(container_rows)
    phase_event_data = pd.DataFrame(event_rows, columns=[
        'filename', 'population', 'gpu_bin', 'event_id', 'phase', 'span_min', 'idle_min', 'unknown_min', 'observed_return'])
    phase_policy_data = pd.DataFrame(policy_rows)
    phase_coverage = pd.DataFrame(coverage_rows)
else:
    phase_containers, phase_event_data, phase_policy_data, phase_coverage = (
        pd.read_csv(phase_paths[name]) for name in ['containers', 'events', 'policy_containers', 'coverage'])
    print('[RESULT] identical-input phase CSV cache used')

phase_actual_counts = phase_coverage.groupby(['population', 'gpu_bin']).size().to_dict()
assert phase_actual_counts == PHASE_EXPECTED_COUNTS, f'Population changed: {phase_actual_counts}'
assert not phase_coverage.filename.duplicated().any()
assert not phase_containers.duplicated(['filename', 'phase']).any()
assert len(phase_containers) == 4 * len(phase_coverage)
assert np.allclose(phase_coverage[['active_min', 'idle_min', 'unknown_min']].sum(axis=1), phase_coverage.span_min, atol=1e-6)
assert np.allclose(phase_containers.groupby('filename').idle_min.sum().sort_index(),
                   phase_coverage.set_index('filename').idle_min.sort_index(), atol=1e-6)
assert np.isclose(phase_event_data.idle_min.sum(), phase_containers.idle_min.sum(), atol=1e-6)

# Reconcile the new after-trigger decomposition with section 6 if its CSV exists.
candidate_file = PHASE_RESULTS / '06_policy_candidate_summary.csv'
if candidate_file.is_file():
    old = pd.read_csv(candidate_file)
    for (pop, group, policy), part in phase_policy_data.groupby(['population', 'gpu_bin', 'policy']):
        engine_bin = 'GPU-Enabled' if group == 'Integer-GPU' else group
        match = old.population.eq(pop) & old.gpu_bin.eq(engine_bin)
        match &= (old.policy.str.startswith('IR2') if policy == 'IR2' else
                  old.pre_timeout_min.eq(150) & old.post_timeout_min.eq(150))
        previous = old.loc[match]
        assert len(previous) == 1
        unique = part.drop_duplicates('filename')
        for resource in ['cpu', 'gpu']:
            assert np.isclose(part[f'idle_{resource}_min'].sum(), previous.iloc[0][f'idle_{resource}_min'], atol=.05, rtol=1e-9)
            assert np.isclose(unique[f'baseline_{resource}_min'].sum(), previous.iloc[0][f'baseline_{resource}_min'], atol=.05, rtol=1e-9)
        assert unique.later_active.sum() == previous.iloc[0].return_containers
    print('[CHECK] 6절 online idle resource-time / denominator / later activity reconciled')
else:
    print('[CHECK] 6절 CSV 없음: phase partition 확인 완료; 기존 표와 직접 대조는 미실행')

for name, data in [('containers', phase_containers), ('events', phase_event_data),
                   ('policy_containers', phase_policy_data), ('coverage', phase_coverage)]:
    data.to_csv(phase_paths[name], index=False, encoding='utf-8-sig', float_format='%.10f')
phase_provenance_path.write_text(json.dumps(phase_inputs, ensure_ascii=False, indent=2))
print('[RESULT] phase calculation PASS; raw phase measurements now available')


### 7.4 전체 idle 시간의 단계별 비중과 분포

In [ ]:
# 7.4 Tables: total composition, per-container tails/concentration, event tails
def phase_with_all(data):
    return pd.concat([data, data.assign(gpu_bin='All')], ignore_index=True)

phase_all = phase_with_all(phase_containers)
phase_total_table = phase_all.groupby(['population', 'gpu_bin', 'phase'], sort=False)[
    ['idle_min', 'idle_cpu_min', 'idle_gpu_min']].sum().reset_index()
for metric in ['idle_min', 'idle_cpu_min', 'idle_gpu_min']:
    denominator = phase_total_table.groupby(['population', 'gpu_bin'])[metric].transform('sum')
    phase_total_table[f'{metric}_share_pct'] = 100 * phase_total_table[metric] / denominator.replace(0, np.nan)
phase_total_table.to_csv(PHASE_RESULTS / '06_idle_phase_totals.csv', index=False, encoding='utf-8-sig')
print('전체 idle 시간: 총량(container·min / core·min / GPU·min) 및 단계별 비중(%)')
display(phase_total_table.round(2))

distribution_rows = []
for (pop, group, phase), part in phase_all.groupby(['population', 'gpu_bin', 'phase'], sort=False):
    for metric, unit in [('idle_min', 'container·min'), ('idle_cpu_min', 'core·min'), ('idle_gpu_min', 'GPU·min')]:
        x = part[metric].to_numpy()
        if metric == 'idle_gpu_min' and group == 'Non-GPU':
            continue
        for scope, selected in [('All containers (zeros included)', x), ('Positive only', x[x > 0])]:
            distribution_rows.append(dict(population=pop, gpu_bin=group, phase=phase,
                metric=metric, unit=unit, scope=scope, containers=len(x), positive_containers=int((x > 0).sum()),
                **phase_quantiles(selected),
                top_1_pct_share=phase_top_share(x, .01), top_5_pct_share=phase_top_share(x, .05),
                top_10_pct_share=phase_top_share(x, .10)))
phase_distribution_table = pd.DataFrame(distribution_rows)
phase_distribution_table.to_csv(PHASE_RESULTS / '06_idle_phase_container_distribution.csv', index=False, encoding='utf-8-sig')
print('컨테이너별 누적 idle 분포 및 집중도 — 표시: container·min / Positive only; 전체 자원별 결과는 CSV')
display(phase_distribution_table.loc[phase_distribution_table.metric.eq('idle_min')
        & phase_distribution_table.scope.eq('Positive only')].round(2))

event_stat_rows = []
event_all = phase_with_all(phase_event_data)
for pop in ['Train', 'Test']:
    for group in ['Non-GPU', 'Integer-GPU', 'All']:
        for phase in PHASE_ORDER:
            part = event_all.loc[event_all.population.eq(pop) & event_all.gpu_bin.eq(group) & event_all.phase.eq(phase)]
            for metric in ['span_min', 'idle_min']:
                event_stat_rows.append(dict(population=pop, gpu_bin=group, phase=phase,
                    event_metric=metric, events=len(part), containers=part.filename.nunique(),
                    unknown_total_min=part.unknown_min.sum(), **phase_quantiles(part[metric])))
phase_event_stats = pd.DataFrame(event_stat_rows)
phase_event_stats.to_csv(PHASE_RESULTS / '06_idle_phase_event_distribution.csv', index=False, encoding='utf-8-sig')
print('Event span 분포(분): Intermediate는 이후 activity return이 관측된 gap; idle support 분포는 CSV')
display(phase_event_stats.loc[phase_event_stats.event_metric.eq('span_min')].round(2))

print('관측 범위 / 누락 시간 확인: unknown을 idle에 포함하지 않음')
display(phase_coverage.groupby(['population', 'gpu_bin'])[
        ['span_min', 'active_min', 'idle_min', 'unknown_min']].sum().round(2))


### 7.5 Online 종료 판단 이후 idle은 어디에서 오는가?

In [ ]:
# 7.5 Attribute online idle resource-time to hindsight lifecycle phases.
policy_all = phase_with_all(phase_policy_data)
phase_policy_table = policy_all.groupby(['population', 'gpu_bin', 'policy', 'phase'], sort=False)[
    ['idle_min', 'idle_cpu_min', 'idle_gpu_min']].sum().reset_index()
policy_base = policy_all.drop_duplicates(['population', 'gpu_bin', 'policy', 'filename']).groupby(
    ['population', 'gpu_bin', 'policy'])[['baseline_cpu_min', 'baseline_gpu_min']].sum().reset_index()
phase_policy_table = phase_policy_table.merge(policy_base, on=['population', 'gpu_bin', 'policy'], validate='many_to_one')
for resource in ['cpu', 'gpu']:
    metric = f'idle_{resource}_min'
    total_idle = phase_policy_table.groupby(['population', 'gpu_bin', 'policy'])[metric].transform('sum')
    phase_policy_table[f'{resource}_share_of_post_trigger_idle_pct'] = 100 * phase_policy_table[metric] / total_idle.replace(0, np.nan)
    phase_policy_table[f'{resource}_pct_of_baseline'] = 100 * phase_policy_table[metric] / phase_policy_table[f'baseline_{resource}_min'].replace(0, np.nan)
phase_policy_table.to_csv(PHASE_RESULTS / '06_idle_phase_policy_attribution.csv', index=False, encoding='utf-8-sig')
print('Integer-GPU: online 최초 종료 판단 이후 idle GPU·min의 단계별 출처')
display(phase_policy_table.loc[phase_policy_table.gpu_bin.eq('Integer-GPU'), [
    'population', 'policy', 'phase', 'idle_gpu_min', 'gpu_share_of_post_trigger_idle_pct', 'gpu_pct_of_baseline']].round(2))
print('Non-GPU: online 최초 종료 판단 이후 idle core·min의 단계별 출처')
display(phase_policy_table.loc[phase_policy_table.gpu_bin.eq('Non-GPU'), [
    'population', 'policy', 'phase', 'idle_cpu_min', 'cpu_share_of_post_trigger_idle_pct', 'cpu_pct_of_baseline']].round(2))

trigger_rows = policy_all.drop_duplicates(['population', 'gpu_bin', 'policy', 'filename'])
phase_trigger_table = trigger_rows.groupby(['population', 'gpu_bin', 'policy', 'trigger_phase']).agg(
    containers=('filename', 'size'), later_active=('later_active', 'sum')).reset_index()
phase_trigger_table.to_csv(PHASE_RESULTS / '06_idle_phase_trigger_counts.csv', index=False, encoding='utf-8-sig')
print('종료를 유발한 단계별 container 수 및 이후 activity 수 — 위 idle 출처 표와 구분')
display(phase_trigger_table.loc[phase_trigger_table.gpu_bin.eq('Integer-GPU')])


In [ ]:
# 7.6 Figures: full idle composition + policy composition, with explicit denominators.
import matplotlib.pyplot as plt

phase_colors = {
    'Pre': '#326a9b',
    'Intermediate': '#b78429',
    'Post': '#74558a',
    'Never-active': '#6b7176'
}

def phase_composition_figure(data, policy=False):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

    for ax, (group, metric, title) in zip(axes, [
        ('All', 'idle_min', 'All: container-min'),
        ('All', 'idle_cpu_min', 'All: CPU core-min'),
        ('Integer-GPU', 'idle_gpu_min', 'Integer-GPU: GPU-min')
    ]):
        subset = data.loc[data.gpu_bin.eq(group)]
        keys = ['population', 'policy'] if policy else ['population']

        table = (
            subset.pivot(index=keys, columns='phase', values=metric)
            .reindex(columns=PHASE_ORDER, fill_value=0)
        )

        order = (
            pd.MultiIndex.from_product(
                [['Train', 'Test'], ['IR2', 'Static 150']],
                names=keys
            )
            if policy else ['Train', 'Test']
        )
        table = table.reindex(order)

        shares = 100 * table.div(
            table.sum(axis=1).replace(0, np.nan),
            axis=0
        )
        bottom = np.zeros(len(shares))

        for phase in PHASE_ORDER:
            vals = shares[phase].fillna(0).to_numpy()
            ax.bar(
                np.arange(len(vals)),
                vals,
                bottom=bottom,
                color=phase_colors[phase],
                label=phase,
                edgecolor='white',
                linewidth=.6
            )

            for i, val in enumerate(vals):
                if val >= 7:
                    ax.text(
                        i, bottom[i] + val / 2,
                        f'{val:.1f}%',
                        ha='center', va='center',
                        color='white', fontsize=10
                    )
            bottom += vals

        ax.set_title(title)
        ax.set_ylim(0, 100)
        ax.set_xticks(
            np.arange(len(shares)),
            [f'{i[0]}\n{i[1]}' for i in shares.index]
            if policy else shares.index
        )
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_axisbelow(True)
        ax.grid(axis='y', color='#dddddd', linewidth=.5)

    axes[0].set_ylabel('Share of observed idle resource-time (%)')
    handles, labels = axes[0].get_legend_handles_labels()

    # 제목 → 범례 → 패널 순서로 배치하고 상단 공간 확보
    fig.suptitle(
        'Idle after first online termination: phase composition'
        if policy else 'Whole-trace observed idle: phase composition',
        y=.98,
        va='top'
    )
    fig.legend(
        handles, labels,
        loc='upper center',
        bbox_to_anchor=(.5, .92),
        bbox_transform=fig.transFigure,
        ncol=4,
        frameon=False
    )
    fig.subplots_adjust(
        left=.065,
        right=.99,
        bottom=.14,
        top=.78,
        wspace=.05
    )

    return fig

fig = phase_composition_figure(phase_total_table)
fig.savefig(
    PHASE_RESULTS / '06_idle_phase_composition.png',
    dpi=180, bbox_inches='tight'
)
plt.show()

fig = phase_composition_figure(phase_policy_table, policy=True)
fig.savefig(
    PHASE_RESULTS / '06_idle_phase_policy_composition.png',
    dpi=180, bbox_inches='tight'
)
plt.show()


### 7.7 IR2의 단계별 이득 가설: paired exposure / 현재 idle episode

In [ ]:
# 7.7 Paired outcomes and trigger-stage trade-offs. No Test threshold tuning.
phase_one_policy = phase_with_all(phase_policy_data.drop_duplicates(['filename', 'policy']))
phase_one_policy['operational_stage'] = phase_one_policy.trigger_phase.map({
    'Pre': 'Before first activity', 'Never-active': 'Before first activity',
    'Intermediate': 'After observed activity', 'Post': 'After observed activity',
    'No trigger': 'No trigger'})
phase_stage_table = phase_one_policy.groupby(['population', 'gpu_bin', 'policy', 'operational_stage']).agg(
    containers=('filename', 'size'), later_active=('later_active', 'sum'),
    current_episode_idle_cpu_min=('episode_idle_cpu_min', 'sum'),
    current_episode_idle_gpu_min=('episode_idle_gpu_min', 'sum'),
    observed_activity_after_cpu_min=('active_after_cpu_min', 'sum'),
    observed_activity_after_gpu_min=('active_after_gpu_min', 'sum')).reset_index()
phase_stage_table['later_active_pct_of_triggered_stage'] = np.where(
    phase_stage_table.operational_stage.ne('No trigger'),
    100 * phase_stage_table.later_active / phase_stage_table.containers, np.nan)
phase_stage_table.to_csv(PHASE_RESULTS / '06_idle_phase_operational_stage_comparison.csv', index=False, encoding='utf-8-sig')
print('GPU: operational Pre / activity 이후 단계별 노출과 현재 idle episode의 회수 시간')
display(phase_stage_table.loc[phase_stage_table.gpu_bin.eq('Integer-GPU')].round(2))
print('Non-GPU: 동일 비교 (GPU와 분리해서 해석)')
display(phase_stage_table.loc[phase_stage_table.gpu_bin.eq('Non-GPU')].drop(columns=[
    'current_episode_idle_gpu_min', 'observed_activity_after_gpu_min']).round(2))

phase_unique = phase_policy_data.drop_duplicates(['filename', 'policy'])
paired_columns = ['filename', 'population', 'gpu_bin', 'trigger_phase', 'later_active', 'terminated', 'trigger_elapsed_min']
phase_paired = phase_unique.loc[phase_unique.policy.eq('IR2'), paired_columns].merge(
    phase_unique.loc[phase_unique.policy.eq('Static 150'), paired_columns],
    on=['filename', 'population', 'gpu_bin'], suffixes=('_ir2', '_static'), validate='one_to_one')
ir2_exposed = phase_paired.later_active_ir2.eq(1)
static_exposed = phase_paired.later_active_static.eq(1)
phase_paired['exposure_outcome'] = np.select([
    ir2_exposed & static_exposed, ir2_exposed & ~static_exposed, ~ir2_exposed & static_exposed],
    ['Both exposed', 'IR2 only exposed', 'Static only exposed'], default='Neither exposed')
phase_paired.to_csv(PHASE_RESULTS / '06_idle_phase_paired_containers.csv', index=False, encoding='utf-8-sig')
phase_paired_all = phase_with_all(phase_paired)
phase_paired_counts = phase_paired_all.groupby(['population', 'gpu_bin', 'exposure_outcome']).size().rename('containers').reset_index()
phase_paired_counts.to_csv(PHASE_RESULTS / '06_idle_phase_paired_exposure_counts.csv', index=False, encoding='utf-8-sig')
print('같은 컨테이너에서 누가 later-active 노출을 추가/회피했는가?')
display(phase_paired_counts)
phase_paired_cross = phase_paired_all.groupby([
    'population', 'gpu_bin', 'trigger_phase_ir2', 'trigger_phase_static', 'exposure_outcome']).size().rename('containers').reset_index()
phase_paired_cross.to_csv(PHASE_RESULTS / '06_idle_phase_paired_trigger_cross.csv', index=False, encoding='utf-8-sig')
print('Integer-GPU: 두 정책의 종료 단계 및 paired exposure 이동')
display(phase_paired_cross.loc[phase_paired_cross.gpu_bin.eq('Integer-GPU')])


### 7.8 결과를 읽는 순서와 제한

## 8. Online / Hindsight의 두 관점에서 IR2–Static 해석
### 8.1 기존 hindsight 회계를 유지한 직접 비교

In [ ]:
# 8.2 Legacy hindsight IR2 / Static 150 comparison; no mutation of calc.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

def legacy_hindsight_contributions(frame, policy):
    required=['filename','population','gpu_bin','container_class','first_active_minute',
              'trace_end_minute','tail_is_idle','post_idle_minute','cpu_allocated(Core)','gpu_allocated(EA)']
    if set(required) - set(frame.columns):
        raise ValueError(f'Missing calc fields: {set(required) - set(frame.columns)}')
    result=frame[required].copy()
    assert result.filename.is_unique
    assert result.gpu_bin.isin(['Non-GPU','Integer-GPU']).all()
    if policy == 'IR2':
        pre=result.gpu_bin.map({'Non-GPU':85,'Integer-GPU':119})  # GPU Pre 119 (Train P90 119.0)
        post=result.gpu_bin.map({'Non-GPU':146,'Integer-GPU':170})
    elif policy == 'Static 150':
        pre=pd.Series(150.0,index=result.index)
        post=pd.Series(150.0,index=result.index)
    else:
        raise ValueError(policy)
    never=result.container_class.eq('low_util')
    normal=result.container_class.eq('normal')
    assert (never | normal).all()
    result['policy']=policy
    result['never_saved_min']=np.where(never,(result.trace_end_minute-pre).clip(lower=0),0.0)
    result['eventually_pre_saved_min']=np.where(normal,
        (result.first_active_minute.fillna(0)-1-pre).clip(lower=0),0.0)
    result['final_post_saved_min']=np.where(normal & result.tail_is_idle.eq(True),
        (result.post_idle_minute.fillna(0)-post).clip(lower=0),0.0)
    result['pre_with_later_activity_candidate']=(result.eventually_pre_saved_min>0).astype(int)
    for resource,alloc in [('cpu','cpu_allocated(Core)'),('gpu','gpu_allocated(EA)')]:
        result[f'{resource}_baseline']=result[alloc]*result.trace_end_minute
        for phase in ['never','eventually_pre','final_post']:
            result[f'{resource}_{phase}_saved']=result[alloc]*result[f'{phase}_saved_min']
        result[f'{resource}_pre_combined_saved']=result[f'{resource}_never_saved']+result[f'{resource}_eventually_pre_saved']
        result[f'{resource}_total_saved']=result[f'{resource}_pre_combined_saved']+result[f'{resource}_final_post_saved']
    return result

if 'calc' not in globals():
    raise RuntimeError('calc가 없습니다. 기존 노트북 1절 입력 셀과 2절 frozen savings 셀을 먼저 실행하세요.')
hindsight_ir2=legacy_hindsight_contributions(calc,'IR2')
hindsight_static=legacy_hindsight_contributions(calc,'Static 150')

# Assert IR2 exactly reproduces existing contributions, when section 2 is present.
for resource in ['cpu','gpu']:
    for existing,new in [('pre_saved','pre_combined_saved'),('post_saved','final_post_saved'),('total_saved','total_saved')]:
        original_col=f'{resource}_{existing}'
        if original_col in calc:
            assert np.allclose(calc[original_col],hindsight_ir2[f'{resource}_{new}'],atol=1e-6), original_col

hindsight_detail=pd.concat([hindsight_ir2,hindsight_static],ignore_index=True)
hindsight_grouped=pd.concat([hindsight_detail,hindsight_detail.assign(gpu_bin='All')],ignore_index=True)
hindsight_rows=[]
for (period,group,policy),part in hindsight_grouped.groupby(['population','gpu_bin','policy'],sort=False):
    resources=['cpu','gpu'] if group=='Integer-GPU' else ['cpu']
    for resource in resources:
        denominator=float(part[f'{resource}_baseline'].sum())
        record=dict(population=period,gpu_bin=group,resource=resource.upper(),
                    unit='GPU·min' if resource=='gpu' else 'core·min',policy=policy,
                    population_containers=len(part),baseline_resource_min=denominator,
                    eventually_pre_candidates=int(part.pre_with_later_activity_candidate.sum()))
        for phase in ['never','eventually_pre','pre_combined','final_post','total']:
            amount=float(part[f'{resource}_{phase}_saved'].sum())
            record[f'{phase}_resource_min']=amount
            record[f'{phase}_pct_baseline']=100*amount/denominator if denominator>0 else np.nan
        hindsight_rows.append(record)
hindsight_summary=pd.DataFrame(hindsight_rows)
assert len(hindsight_summary)==16
assert hindsight_summary.groupby(['population','gpu_bin','resource']).baseline_resource_min.nunique().eq(1).all()

comparison_keys=['population','gpu_bin','resource','unit']
metrics=['never_resource_min','eventually_pre_resource_min','pre_combined_resource_min',
         'final_post_resource_min','total_resource_min','total_pct_baseline','eventually_pre_candidates']
hindsight_comparison=hindsight_summary.loc[hindsight_summary.policy.eq('IR2'),comparison_keys+metrics].merge(
    hindsight_summary.loc[hindsight_summary.policy.eq('Static 150'),comparison_keys+metrics],
    on=comparison_keys,suffixes=('_ir2','_static'),validate='one_to_one')
for phase in ['never','eventually_pre','pre_combined','final_post','total']:
    ir2=hindsight_comparison[f'{phase}_resource_min_ir2']
    static=hindsight_comparison[f'{phase}_resource_min_static']
    hindsight_comparison[f'{phase}_difference_resource_min']=ir2-static
    hindsight_comparison[f'{phase}_relative_change_pct']=100*(ir2-static)/static.replace(0,np.nan)
hindsight_comparison['total_difference_pp']=hindsight_comparison.total_pct_baseline_ir2-hindsight_comparison.total_pct_baseline_static
hindsight_comparison['additional_eventually_pre_candidates']=hindsight_comparison.eventually_pre_candidates_ir2-hindsight_comparison.eventually_pre_candidates_static

hindsight_results=RESULTS_REVISION
hindsight_results.mkdir(parents=True,exist_ok=True)
for name,data in [('container_contributions',hindsight_detail),('summary',hindsight_summary),('ir2_static_comparison',hindsight_comparison)]:
    data.to_csv(hindsight_results/f'06_hindsight_{name}.csv',index=False,encoding='utf-8-sig',float_format='%.10f')
print('[RESULT] Hindsight comparison: same original phase accounting / same denominator')
labels={'population':'Period / 기간','gpu_bin':'Population / 집단','policy':'Policy / 정책',
        'resource':'Resource / 자원','unit':'Unit / 단위',
        'never_resource_min':'Never-active saved / 회수 시간',
        'eventually_pre_resource_min':'Eventually-active Pre saved / 회수 시간',
        'pre_combined_resource_min':'Pre + Never-active saved / 회수 시간',
        'final_post_resource_min':'Final/Post saved / 회수 시간',
        'total_pct_baseline':'Total, % / 전체 할당 대비 회수율',
        'eventually_pre_candidates':'Eventually-active Pre candidates / 이후 활동 후보 수'}
display(hindsight_summary[['population','gpu_bin','resource','unit','policy',
    'never_resource_min','eventually_pre_resource_min','pre_combined_resource_min',
    'final_post_resource_min','total_pct_baseline','eventually_pre_candidates']].rename(columns=labels).round(2))
print('IR2 relative change (%) vs Static — phase-separated; population exposure counts are separate')
display(hindsight_comparison[comparison_keys+[
    'never_relative_change_pct','eventually_pre_relative_change_pct','pre_combined_relative_change_pct',
    'final_post_relative_change_pct','total_relative_change_pct','total_difference_pp',
    'additional_eventually_pre_candidates']].round(2))
print('Result: PASS (legacy IR2 contributions reproduced; policy denominators matched)')
print('Intermediate: excluded from this hindsight accounting, not a measured zero-risk result.')


### 8.3 두 관점을 합친 해석

## 9. Online에서 IR2가 Static보다 두 지표를 동시에 개선하는 지점

In [ ]:
# 9.1 All 23 single-static candidates: online joint resource/exposure comparison.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

def online_joint_comparison(candidates):
    rows=[]
    for period in ['Train','Test']:
        for group,resource in [('Non-GPU','cpu'),('GPU-Enabled','gpu'),('GPU-Enabled','cpu'),('All','cpu')]:
            part=candidates.loc[candidates.population.eq(period)&candidates.gpu_bin.eq(group)]
            reference=part.loc[part.policy_family.eq('IR2')]
            assert len(reference)==1
            reference=reference.iloc[0]
            static=part.loc[part.policy_family.eq('Single static')].sort_values('pre_timeout_min')
            assert len(static)>0 and static.pre_timeout_min.eq(static.post_timeout_min).all()
            for _,point in static.iterrows():
                baseline=float(point[f'baseline_{resource}_min'])
                assert np.isclose(baseline,reference[f'baseline_{resource}_min']) and baseline>0
                assert int(point.containers)==int(reference.containers)
                delta=float(reference[f'idle_{resource}_min']-point[f'idle_{resource}_min'])
                exposure_delta=int(reference.return_containers-point.return_containers)
                # Only numerical equality tolerance, no practical superiority margin.
                equal_amount=np.isclose(delta,0,atol=1e-6,rtol=0)
                ir2_dominates=(delta>=-1e-6 and exposure_delta<=0 and (not equal_amount or exposure_delta<0))
                static_dominates=(delta<=1e-6 and exposure_delta>=0 and (not equal_amount or exposure_delta>0))
                verdict=('IR2 dominates' if ir2_dominates else 'Static dominates' if static_dominates
                         else 'Equal' if equal_amount and exposure_delta==0 else 'Trade-off')
                rows.append(dict(population=period,gpu_bin='Integer-GPU' if group=='GPU-Enabled' else group,
                    resource=resource.upper(),static_timeout_min=int(point.pre_timeout_min),
                    ir2_idle_pct=100*reference[f'idle_{resource}_min']/baseline,
                    static_idle_pct=100*point[f'idle_{resource}_min']/baseline,
                    ir2_resource_difference_min=delta,ir2_resource_difference_pp=100*delta/baseline,
                    ir2_relative_resource_change_pct=100*delta/point[f'idle_{resource}_min'] if point[f'idle_{resource}_min']>0 else np.nan,
                    ir2_later_active=int(reference.return_containers),static_later_active=int(point.return_containers),
                    ir2_exposure_difference=exposure_delta,verdict=verdict))
    return pd.DataFrame(rows)

online_results=RESULTS_REVISION
online_input=online_results/'06_static_timeout_comparison.csv'
if not online_input.is_file():
    raise FileNotFoundError(f'6절에서 생성한 전체 timeout CSV가 필요합니다: {online_input}')
online_candidates=pd.read_csv(online_input)
online_joint=online_joint_comparison(online_candidates)
assert len(online_joint)==184, 'Expected 23 timeouts × 4 metrics × 2 periods'
online_joint.to_csv(online_results/'06_online_joint_comparison.csv',index=False,encoding='utf-8-sig')
online_repeat=online_joint.pivot(index=['gpu_bin','resource','static_timeout_min'],columns='population',values='verdict').reset_index()
online_repeat['ir2_dominates_both_periods']=online_repeat.Train.eq('IR2 dominates') & online_repeat.Test.eq('IR2 dominates')
online_repeat.to_csv(online_results/'06_online_joint_train_test_consistency.csv',index=False,encoding='utf-8-sig')
print('[RESULT] Online IR2/static joint comparison: ALL candidates classified')
print('수치상 IR2가 회수량·노출 두 지표를 함께 개선한 지점:')
display(online_joint.loc[online_joint.verdict.eq('IR2 dominates')].round(3))
print('Train/Test 양쪽에서 반복된 우위 지점:')
display(online_repeat.loc[online_repeat.ir2_dominates_both_periods])
print('비교 위치 확인: 105 / 120 / 135 / 150 / 170 / 180분 — 전체 결과는 CSV')
display(online_joint.loc[online_joint.static_timeout_min.isin([105,120,135,150,170,180])].round(3))
print('Result: PASS (policy denominators / populations matched; all 184 comparisons reported)')
print('Exploratory numerical dominance; not statistical significance or actual user-harm reduction.')
